<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0;">🔍 Lab 04: Trace Your LangGraph Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">See graph-level visibility — node transitions, edge evaluations, and state changes alongside model and tool spans</p>
</div>

**What We Learn:** How LangGraph tracing gives you graph-level visibility — seeing node transitions, edge evaluations, and state changes alongside the standard model and tool call spans.

---


## 🧳 The Contoso Travel Story

Debugging a graph-based agent means understanding not just what the model said, but which nodes executed, which edges were taken, and how state evolved at each step.

- **The Problem:** When a multi-tool query produces unexpected results, the team needs to see which graph nodes ran, which conditional edges were evaluated, and how the message state changed at each step.
- **This Lab Solves:** Configuring tracing for LangGraph agents — where you get graph-topology spans (node entry/exit, edge routing) in addition to the standard model and tool spans.

## 1. Setup

---


In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"✅ Connected to Microsoft Foundry")
print(f"   Model: {model_name}")

In [ ]:
from typing import Optional
from langchain_core.tools import tool
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# Load Contoso Travel data
data_path = "../../data/contoso-travel"
flights_df = pd.read_csv(f"{data_path}/flights.csv")
hotels_df = pd.read_csv(f"{data_path}/hotels.csv")
cars_df = pd.read_csv(f"{data_path}/car_rentals.csv")


@tool
def search_flights(origin: Optional[str] = None, destination: Optional[str] = None) -> str:
    """Search for available Contoso Travel flights by origin and/or destination city."""
    results = flights_df.copy()
    if origin:
        results = results[results["origin"].str.lower() == origin.lower()]
    if destination:
        results = results[results["destination"].str.lower() == destination.lower()]
    if results.empty:
        return json.dumps({"message": "No flights found.", "results": []})
    return results.to_json(orient="records")


@tool
def search_hotels(city: Optional[str] = None, max_price: Optional[float] = None) -> str:
    """Search for available Contoso Travel hotels by city and optional max price per night."""
    results = hotels_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if max_price:
        results = results[results["price_per_night_usd"] <= max_price]
    if results.empty:
        return json.dumps({"message": "No hotels found.", "results": []})
    return results.to_json(orient="records")


@tool
def search_car_rentals(city: Optional[str] = None, car_type: Optional[str] = None) -> str:
    """Search for available Contoso Travel car rentals by city and optional car type."""
    results = cars_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if car_type:
        results = results[results["car_type"].str.lower() == car_type.lower()]
    results = results[results["available"] == True]
    if results.empty:
        return json.dumps({"message": "No car rentals found.", "results": []})
    return results.to_json(orient="records")


tools = [search_flights, search_hotels, search_car_rentals]

# Create Azure OpenAI model
llm = AzureChatOpenAI(
    azure_deployment=model_name,
    azure_endpoint=os.environ["MODEL_ENDPOINT"],
    api_key=os.environ["MODEL_API_KEY"],
    api_version="2024-12-01-preview",
)
llm_with_tools = llm.bind_tools(tools)


# Build LangGraph agent
def agent_node(state: MessagesState):
    """Call the model and return updated messages."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")

app = graph.compile()

print("✅ LangGraph agent compiled with 3 Contoso Travel tools")
print(f"   Tools: {[t.name for t in tools]}")
print(f"   Graph: START → agent → tools_condition → [tools → agent | END]")

## 2. How LangGraph Tracing Differs

Tracing looks different depending on how you built your agent:

- **Prompted agents:** Flat spans — model call, tool call, model call. You see what the model did, but not the control flow around it.
- **MAF agents:** Custom spans you add manually, plus auto-instrumented HTTP/model/tool spans. You control what gets traced.
- **LangGraph agents:** Graph-topology spans appear automatically — node entry/exit, edge evaluation, state transitions. Each node in your graph becomes a span, and edges show routing decisions.

| Trace Shape | Prompted | MAF | LangGraph |
|-------------|----------|-----|-----------|
| Model calls | ✅ | ✅ | ✅ |
| Tool calls | ✅ | ✅ | ✅ |
| HTTP spans | ❌ | ✅ (auto) | ✅ (auto) |
| Custom business logic | ❌ | ✅ (manual) | ✅ (per node) |
| Graph topology | ❌ | ❌ | ✅ (auto) |
| State transitions | ❌ | ❌ | ✅ (auto) |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Key insight:</b> With LangGraph, you don't need to manually instrument your control flow — the graph structure <i>is</i> the instrumentation.
</div>

---


<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #533483; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 20px 0;">
  <h2 style="margin: 0;">Part A: Console Tracing</h2>
</div>

---


In [ ]:
import os
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

print("✅ Console tracing enabled — spans will print below")

## 4. Run a Traced Query

Invoke the LangGraph agent and watch the console output below. You'll see spans for each graph node (`agent`, `tools`), edge evaluations (`tools_condition`), and the underlying model and tool calls.

---


In [ ]:
from langchain_core.messages import HumanMessage

result = app.invoke(
    {"messages": [HumanMessage(content="Find flights from Seattle to Paris and a hotel in Paris under $200/night")]},
)

print(f"🤖 Agent Response:\n{result['messages'][-1].content}")

In the console output above, you should see trace spans showing the graph topology:

- **`agent`** node — The model receives the query and decides to call tools
- **`tools_condition`** edge — Evaluates whether the model requested tool calls
- **`tools`** node — Executes `search_flights` and `search_hotels`
- **`agent`** node (again) — The model receives tool results and generates the final response
- **`tools_condition`** edge (again) — Evaluates to `END` since no more tool calls are needed

Each span includes `trace_id`, `span_id`, `parent_id`, and attributes like model name and token counts.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Tip:</b> Compare this with the flat trace from Lab 04 in the Prompted track — there you only see model and tool spans, not the graph routing decisions.
</div>

---


<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #533483; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 20px 0;">
  <h2 style="margin: 0;">Part B: Azure Monitor Tracing</h2>
</div>

---


In [ ]:
# Shut down console tracer to avoid duplicate spans
provider.shutdown()

from azure.monitor.opentelemetry import configure_azure_monitor

conn_string = project_client.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=conn_string)

tracer = trace.get_tracer("contoso-travel-langgraph")

print("✅ Azure Monitor tracing enabled")
print(f"   Traces will flow to Application Insights")
print(f"   View them in the Foundry portal → Tracing tab")

In [ ]:
with tracer.start_as_current_span("contoso-travel-lg-query") as span:
    span.set_attribute("travel.query_type", "multi_search")
    span.set_attribute("travel.framework", "langgraph")

    result = app.invoke(
        {"messages": [HumanMessage(content="I need a flight from NYC to London and a car rental in London")]},
    )

    print(f"🤖 Agent Response:\n{result['messages'][-1].content}")
    print(f"\n📊 Trace sent to Azure Monitor!")
    print(f"   Trace ID: {span.get_span_context().trace_id:032x}")

## 6. View in Foundry Portal

To view your LangGraph traces:

1. Go to the [Microsoft Foundry portal](https://ai.azure.com)
2. Open your project
3. Click on the **Tracing** tab in the left navigation
4. Find your trace by the span name `contoso-travel-lg-query`
5. Expand the trace tree to see the graph-specific spans

In the trace view, you'll see how LangGraph spans differ from prompted-agent traces:
- **Node spans** show each graph node's execution (`agent`, `tools`)
- **Edge spans** show routing decisions (`tools_condition` → `tools` or `END`)
- **State transitions** show how the message list grew at each step

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #9c27b0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>⏱️ Note:</b> Traces may take 2–5 minutes to appear in Azure Monitor after execution.
</div>

---


## 7. Compare Trace Shapes

Here's how the same multi-tool query looks in each track's trace view:

### Prompted (flat)
```
contoso-travel-query
  └─ responses.create
       ├─ model call → decides to call tools
       ├─ tool: search_flights
       ├─ tool: search_hotels
       └─ model call → final response
```

### MAF (custom spans)
```
contoso-travel-query
  ├─ [custom] flight-search
  │    └─ HTTP POST → model endpoint
  ├─ [custom] hotel-search
  │    └─ HTTP POST → model endpoint
  └─ [custom] aggregate-results
       └─ HTTP POST → model endpoint
```

### LangGraph (graph topology)
```
contoso-travel-lg-query
  └─ StateGraph
       ├─ node: agent (model call → tool requests)
       ├─ edge: tools_condition → "tools"
       ├─ node: tools
       │    ├─ search_flights()
       │    └─ search_hotels()
       ├─ node: agent (model call → final response)
       └─ edge: tools_condition → END
```

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Key insight:</b> LangGraph traces are the richest because the graph structure itself becomes visible in the trace — you can see exactly which path the agent took through the graph.
</div>

---


## 8. Cleanup

---


In [ ]:
project_client.close()
credential.close()
print("✅ Clients closed")

## 9. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Configured <b>console tracing</b> for a LangGraph agent with <code>ConsoleSpanExporter</code></li>
  <li>Observed <b>graph-topology spans</b> — node entry/exit, edge routing, state transitions</li>
  <li>Configured <b>Azure Monitor tracing</b> for production observability using <code>configure_azure_monitor</code></li>
  <li>Compared trace shapes across <b>Prompted</b>, <b>MAF</b>, and <b>LangGraph</b> implementations</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>💡 Key Takeaway:</b> LangGraph tracing gives you the richest observability of any implementation approach. The graph structure itself becomes visible — you can see exactly which nodes executed, which edges were taken, and how the agent's state evolved at each step.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>➡️ Next:</b> In <a href="lab-05-evaluation.ipynb">Lab 05</a>, we'll <b>evaluate</b> the LangGraph agent for quality and safety — confirming that evaluation is truly agent-agnostic!
</div>

---
